# Patches — Bloc 5a, Sessió 8
### Síntesi amb classes: Oscillator, Envelope, SamplePlayer

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

Aquest notebook és per a la **demo guiada en directe**.

In [ ]:
import numpy as np
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

sample_rate = 44100

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/05_sintesi/sessio08_oscil_ladors_classes/recursos/audio/"
urllib.request.urlretrieve(base_url + "kick_sample.wav", "kick_sample.wav")
print("Fitxers carregats")

## 1. La classe Oscillator

In [ ]:
class Oscillator:
    def __init__(self, freq=440.0, waveform='sine', sample_rate=44100):
        self.freq = freq
        self.waveform = waveform
        self.sample_rate = sample_rate

    def set_freq(self, freq):
        self.freq = freq

    def generate(self, duration):
        n = int(self.sample_rate * duration)
        t = np.linspace(0, duration, n, endpoint=False)

        if self.waveform == 'sine':
            return np.sin(2 * np.pi * self.freq * t)
        elif self.waveform == 'square':
            return signal.square(2 * np.pi * self.freq * t)
        elif self.waveform == 'sawtooth':
            return signal.sawtooth(2 * np.pi * self.freq * t)
        else:
            raise ValueError(f"Forma d'ona desconeguda: {self.waveform}")

osc = Oscillator(freq=440, waveform='sine')
wave = osc.generate(duration=2.0)
print("Generat:", len(wave), "mostres")
Audio(wave, rate=sample_rate)

🎤 *Com sona? Té atac suau? Té final suau?*

In [ ]:
plt.figure(figsize=(10,2))
plt.plot(wave[:2000])
plt.title("Inici de l'ona — entra de cop")
plt.show()

plt.figure(figsize=(10,2))
plt.plot(wave[-2000:])
plt.title("Final de l'ona — surt de cop")
plt.show()

## 2. La classe Envelope (ADSR)

In [ ]:
class Envelope:
    def __init__(self, attack=0.01, decay=0.1, sustain=0.7, release=0.2):
        self.attack = attack
        self.decay = decay
        self.sustain = sustain
        self.release = release

    def generate(self, note_duration, sample_rate=44100):
        n_attack  = int(self.attack  * sample_rate)
        n_decay   = int(self.decay   * sample_rate)
        n_sustain = max(0, int(note_duration * sample_rate) - n_attack - n_decay)
        n_release = int(self.release * sample_rate)

        env_attack  = np.linspace(0, 1, n_attack)
        env_decay   = np.linspace(1, self.sustain, n_decay)
        env_sustain = np.full(n_sustain, self.sustain)
        env_release = np.linspace(self.sustain, 0, n_release)

        return np.concatenate([env_attack, env_decay, env_sustain, env_release])

env = Envelope(attack=0.05, decay=0.15, sustain=0.6, release=0.3)
env_curve = env.generate(note_duration=0.5)

plt.figure(figsize=(10,3))
plt.plot(env_curve)
plt.title("Envolvent ADSR")
plt.xlabel("mostres")
plt.ylabel("amplitud")
plt.show()

## 3. Combinar Oscillator + Envelope

In [ ]:
osc = Oscillator(freq=440, waveform='sine')
env = Envelope(attack=0.02, decay=0.1, sustain=0.6, release=0.3)

note_duration = 0.5
total_duration = note_duration + env.release

wave = osc.generate(total_duration)
env_curve = env.generate(note_duration)

n = min(len(wave), len(env_curve))
nota_final = wave[:n] * env_curve[:n]

plt.figure(figsize=(10,3))
plt.plot(nota_final)
plt.title("Oscillator x Envelope — ara SÍ té forma de nota")
plt.show()

print("Sense envolvent:")
display(Audio(wave[:n], rate=sample_rate))
print("Amb envolvent:")
display(Audio(nota_final, rate=sample_rate))

## 4. MIDI → freqüència

In [ ]:
def note_to_freq(note_number):
    return 440.0 * (2 ** ((note_number - 69) / 12))

for note in [60, 64, 67, 72]:
    print(f"Nota MIDI {note} -> {note_to_freq(note):.2f} Hz")

## 5. DEMO — una funció que toca una nota MIDI completa

In [ ]:
def tocar_nota(note_number, note_duration, waveform='sine',
                attack=0.02, decay=0.1, sustain=0.6, release=0.2):
    freq = note_to_freq(note_number)
    osc = Oscillator(freq=freq, waveform=waveform)
    env = Envelope(attack=attack, decay=decay, sustain=sustain, release=release)

    total_duration = note_duration + release
    wave = osc.generate(total_duration)
    env_curve = env.generate(note_duration)

    n = min(len(wave), len(env_curve))
    return wave[:n] * env_curve[:n]

# Un petit arpegi (reaprofitant la idea del Bloc 4!)
acord = [60, 64, 67, 72]
trossos = [tocar_nota(n, note_duration=0.2) for n in acord]
sequencia = np.concatenate(trossos)

Audio(sequencia, rate=sample_rate)

## 6. La classe SamplePlayer — un component diferent

In [ ]:
class SamplePlayer:
    def __init__(self, filepath, sample_rate=44100):
        self.sample_rate = sample_rate
        data, sr = sf.read(filepath)
        self.sample = data

    def play(self, gain=1.0):
        return self.sample * gain

kick = SamplePlayer('kick_sample.wav')

print("Oscillator (genera matematicament):")
display(Audio(tocar_nota(36, note_duration=0.2), rate=sample_rate))

print("SamplePlayer (reprodueix un so ja gravat):")
display(Audio(kick.play(), rate=sample_rate))

## 7. DEMO — combinar oscil·lador sintetitzat + sample

In [ ]:
# Un patro ritmic: kick (sample) + nota greu sintetitzada alhora
bass_note = tocar_nota(36, note_duration=0.25, waveform='sawtooth',
                       attack=0.005, decay=0.05, sustain=0.5, release=0.1)
kick_hit = kick.play(gain=0.8)

n = min(len(bass_note), len(kick_hit))
combo = bass_note[:n] + kick_hit[:n]
combo = combo / np.max(np.abs(combo))

Audio(combo, rate=sample_rate)